# Per-Token KL Divergence Analysis: Reasoning-Aligned Controllability Dynamics

### Teacher-Forced Trajectory Divergence: Base vs. SFT Across Exact Reasoning Boundaries

This notebook evaluates **per-token Kullback–Leibler (KL) divergence** between an aligned SFT policy $\pi_{\text{SFT}}$ and an untouched base policy $\pi_{\text{Base}}$.

---

### 1. What Teacher-Forced Forward KL Measures

Given prompt $\bm{x}$ (containing system formatting constraints) and auto-regressive trajectory $\bm{y} = (y_1, y_2, \dots, y_T)$ generated by the compliant SFT model, we compute the forward KL divergence at every token position $t$:

$$D_{\text{KL}}\big(\pi_{\text{SFT}}(\cdot \mid \bm{x}, \bm{y}_{<t}) \;\parallel\; \pi_{\text{Base}}(\cdot \mid \bm{x}, \bm{y}_{<t})\big) = \sum_{w \in \mathcal{V}} \pi_{\text{SFT}}(w \mid \bm{x}, \bm{y}_{<t}) \log \left( \frac{\pi_{\text{SFT}}(w \mid \bm{x}, \bm{y}_{<t})}{\pi_{\text{Base}}(w \mid \bm{x}, \bm{y}_{<t})} \right)$$

#### Precise Interpretation:
1. **Shared Conditioning**: Both models are evaluated under the **exact same conditioning history** $\bm{x} + \bm{y}_{<t}$.
2. **Low KL ($D_{\text{KL}} \approx 0$)**: Conditional on $\bm{y}_{<t}$, Base predictions closely track SFT. Under teacher forcing, this identifies regions of conditional predictive alignment. It does **not** prove that an unguided prefix has "washed out" in free autonomous generation, nor does it unilaterally establish the best prefix cutoff. Instead, it identifies **candidate prefix lengths** to test via empirical intervention sweeps.
3. **High KL ($D_{\text{KL}} \gg 0$)**: Conditional on $\bm{y}_{<t}$, Base assigns substantially different probability mass over the vocabulary than SFT.

---

### 2. Exact Reasoning Boundaries & Prompt State

Models generating Chain-of-Thought reasoning demarcate phases via `<think>...</think>` tags:
* **Prompt Reasoning State**: If the prompt already terminates with `<think>` (common in thinking-model templates), generation begins *immediately* inside reasoning.
* **Opening Tag in Generation**: If `<think>` is produced by the model, reasoning begins immediately *after* `<think>`.
* **Closing Tag**: Reasoning stops **strictly before** `</think>`. Tokens from `</think>` onwards belong to the final answer and are excluded from the reasoning divergence window.

---

### 3. Quadrant Stratification & Task Independence

Records are partitioned into four empirical outcome quadrants:
* **Group 1: SFT=PASS, Base=FAIL (The Focal Signal)**: SFT satisfied the constraint; Base failed.
* **Group 2: SFT=PASS, Base=PASS (Concordant Pass)**: Both models satisfied the constraint; divergence reflects residual stylistic/behavioral differences on tasks Base already handled.
* **Group 3: SFT=FAIL, Base=FAIL (Concordant Fail)**: Intrinsic task difficulty ceiling.
* **Group 4: SFT=FAIL, Base=PASS (SFT Regressions)**: Cases where fine-tuning degraded capabilities present in Base.

*Statistical Note for Haskins*: Haskins evaluates 50 prompts repeated across 10 tasks ($50 \times 10 = 500$). Pooling all 500 samples as independent observations violates independence. This notebook prioritizes **descriptive trajectory curves** and **per-task breakdowns**.

## 1. Environment & Numerical Precision
Verify GPU availability, compute capabilities, and ensure all logit operations and KL summations are performed strictly in **`float32`** to avoid catastrophic cancellation errors.

In [10]:
import os
import gc
import json
import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Publication styling
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 150
})

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Supported:   {torch.cuda.is_bf16_supported()}")


Execution Device: cuda
GPU Model:        NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM Available:   94.97 GB
BF16 Supported:   True


## 2. Configuration & Model Selectors
Select the architecture (`'qwen3_14b'` or `'phi4'`) and dataset (`'haskins_500'` or `'reasonif_300'`).
The configuration explicitly specifies the real base model checkpoints, revisions, and corresponding LoRA adapters.

* **`EXECUTE_FORWARD_PASSES`**:
  * Set `True` on a GPU machine to execute dual-policy forward passes and record measured KL trajectories.
  * Set `False` to load previously measured and verified empirical trajectories from disk.
  * *Note: No synthetic curves or predetermined analytical formulas are used.*

In [11]:
# ==============================================================================
# CONFIGURATION TOGGLE
# ==============================================================================
MODEL_FAMILY = "phi4"       # 'qwen3_14b' or 'phi4'
DATASET_NAME = "reasonif_300"     # 'haskins_500' or 'reasonif_300'
EXECUTE_FORWARD_PASSES = False   # Set True on GPU VM; False to load saved empirical measurements
MAX_REASONING_TOKENS = 512       # Maximum reasoning window tokens to evaluate

MODEL_SPECS = {
    "qwen3_14b": {
        "base_model_id": "Qwen/Qwen3-14B",
        "base_model_revision": "main",
        "lora_adapter_id": "kyleliu789/qwen3-14b-gpt52-high-reasoning-original",
        "lora_adapter_revision": "main",
        "haskins_base": [
            "cot_control/qwen_3_14b/haskins_vllm_runs/Qwen__Qwen3-14B_final/results.jsonl",
            "/home/kyleliu789/uncontaminated_workspace/cot_control/qwen_3_14b/haskins_vllm_runs/Qwen__Qwen3-14B_final/results.jsonl",
            "workspace/eval_haskins_cot_control/cot_obfuscation_code/cot_controllability/results_qwen_comparison/base_scored.jsonl",
            "cot_obfuscation_code/cot_controllability/results_qwen_comparison/base_scored.jsonl",
            "results_qwen_comparison/base_scored.jsonl",
            "base_scored.jsonl"
        ],
        "haskins_sft": [
            "cot_control/qwen_3_14b/haskins_vllm_runs/Qwen__Qwen3-14B__qwen3-14b-gpt52-high-reasoning-original__f9ffa41a9bb9a912/results.jsonl",
            "/home/kyleliu789/uncontaminated_workspace/cot_control/qwen_3_14b/haskins_vllm_runs/Qwen__Qwen3-14B__qwen3-14b-gpt52-high-reasoning-original__f9ffa41a9bb9a912/results.jsonl",
            "workspace/eval_haskins_cot_control/cot_obfuscation_code/cot_controllability/results_qwen_comparison/gpt52_short_scored.jsonl",
            "cot_obfuscation_code/cot_controllability/results_qwen_comparison/gpt52_short_scored.jsonl",
            "results_qwen_comparison/gpt52_short_scored.jsonl",
            "gpt52_short_scored.jsonl"
        ],
        "reasonif_base": [
            "outputs/reasonif_qwen3_14b_base/scored_responses.jsonl",
            "/home/kyleliu789/workspace/outputs/reasonif_qwen3_14b_base/scored_responses.jsonl",
            "workspace/outputs/reasonif_qwen3_14b_base/scored_responses.jsonl",
            "data/reasonif_qwen_results/base/scored_responses.jsonl",
            "scored_responses.jsonl"
        ],
        "reasonif_sft": [
            "outputs/reasonif_qwen3_14b_final_paper/qwen3_14b_gpt52_high_reasoning_original/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "/home/kyleliu789/workspace/outputs/reasonif_qwen3_14b_final_paper/qwen3_14b_gpt52_high_reasoning_original/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "workspace/outputs/reasonif_qwen3_14b_final_paper/qwen3_14b_gpt52_high_reasoning_original/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "data/reasonif_qwen_results/sft/scored_responses.jsonl",
            "scored_responses.jsonl"
        ]
    },
    "phi4": {
        "base_model_id": "microsoft/Phi-4-reasoning",
        "base_model_revision": "main",
        "lora_adapter_id": "kyleliu789/phi4-reasoning-14b-gpt52-high-reasoning-original",
        "lora_adapter_revision": "main",
        "haskins_base": [
            "cot_control/phi/haskins_phi4_runs/microsoft__Phi-4-reasoning__base__vllm__fd600c81f7cb93a6/results.jsonl",
            "/home/kyleliu789/uncontaminated_workspace/cot_control/phi/haskins_phi4_runs/microsoft__Phi-4-reasoning__base__vllm__fd600c81f7cb93a6/results.jsonl",
            "workspace/eval_haskins_cot_control/cot_obfuscation_code/cot_controllability/results_phi4_reasoning_comparison/base_raw.jsonl",
            "cot_obfuscation_code/cot_controllability/results_phi4/base_scored.jsonl",
            "results_phi4/base_scored.jsonl"
        ],
        "haskins_sft": [
            "cot_control/phi/haskins_phi4_runs/microsoft__Phi-4-reasoning__phi4-reasoning-14b-gpt52-high-reasoning-original__vllm__8a9ed1e6b2180ab7/results.jsonl",
            "/home/kyleliu789/uncontaminated_workspace/cot_control/phi/haskins_phi4_runs/microsoft__Phi-4-reasoning__phi4-reasoning-14b-gpt52-high-reasoning-original__vllm__8a9ed1e6b2180ab7/results.jsonl",
            "workspace/eval_haskins_cot_control/cot_obfuscation_code/cot_controllability/results_phi4_reasoning_comparison/lora_raw.jsonl",
            "cot_obfuscation_code/cot_controllability/results_phi4/sft_scored.jsonl",
            "results_phi4/sft_scored.jsonl"
        ],
        "reasonif_base": [
            "outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_base/full_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "/home/kyleliu789/workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_base/full_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_base/full_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "data/reasonif_phi4_results/base/scored_responses.jsonl",
            "scored_responses.jsonl"
        ],
        "reasonif_sft": [
            "outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "/home/kyleliu789/workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl",
            "data/reasonif_phi4_results/sft/scored_responses.jsonl",
            "scored_responses.jsonl"
        ]
    }
}

spec = MODEL_SPECS[MODEL_FAMILY]
print(f"Architecture:    {MODEL_FAMILY}")
print(f"Benchmark:       {DATASET_NAME}")
print(f"Base Model ID:   {spec['base_model_id']}")
print(f"LoRA Adapter ID: {spec['lora_adapter_id']}")


Architecture:    phi4
Benchmark:       reasonif_300
Base Model ID:   microsoft/Phi-4-reasoning
LoRA Adapter ID: kyleliu789/phi4-reasoning-14b-gpt52-high-reasoning-original


## 3. Strict Record Pairing & Provenance Verification
1. **Duplicate Key Prevention**: Raises `ValueError` if duplicate keys exist in Base or SFT records.
2. **Complete Matching**: Raises `ValueError` if any record is missing between Base and SFT.
3. **Saved Token ID Preservation**: If `prompt_token_ids` and `generated_token_ids` exist in `s_item`, they are copied directly. Re-tokenization is used only when token IDs are absent, under explicit verification of rendered chat prompts.

In [12]:
def resolve_path(path_or_candidates) -> Path:
    if isinstance(path_or_candidates, (str, Path)):
        candidates = [path_or_candidates]
    else:
        candidates = list(path_or_candidates)
        
    search_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/home/kyleliu789"),
        Path("/home/kyleliu789/uncontaminated_workspace"),
        Path("/home/kyleliu789/uncontaminated_workspace/cot_control"),
        Path("/home/kyleliu789/workspace"),
        Path("/home/kyleliu789/workspace/eval_haskins_cot_control"),
        Path("/home/kyleliu789/workspace/outputs"),
        Path("C:/Users/bryan"),
        Path("C:/Users/bryan/Downloads"),
        Path("C:/Users/bryan/.gemini/antigravity/brain/535398ee-4974-476d-bc8f-8df8ae245f1d")
    ]
    
    tried = []
    for cand in candidates:
        cand_p = Path(cand)
        if cand_p.is_absolute() and cand_p.exists():
            return cand_p.resolve()
        for root in search_roots:
            test_p = root / cand_p
            tried.append(str(test_p))
            if test_p.exists():
                return test_p.resolve()
            test_name_p = root / cand_p.name
            if test_name_p.exists():
                return test_name_p.resolve()
                
    raise FileNotFoundError(f"Required data file not found from candidates: {candidates}. Searched {len(tried)} paths.")

def extract_compliance_label(record: Dict[str, Any]) -> bool:
    """Extract ground-truth compliance label prioritizing score_equals_one."""
    if "score_equals_one" in record:
        return bool(record["score_equals_one"] == 1)
    if "instruction_following" in record:
        return bool(record["instruction_following"])
    if "compliant_binary" in record:
        return bool(record["compliant_binary"] == 1)
    if "compliance" in record:
        return bool(record["compliance"] >= 0.999)
    raise KeyError(f"No recognized compliance field found in record: {list(record.keys())}")

# Resolve file paths
if DATASET_NAME == "haskins_500":
    base_path = resolve_path(spec["haskins_base"])
    sft_path = resolve_path(spec["haskins_sft"])
else:
    base_path = resolve_path(spec["reasonif_base"])
    sft_path = resolve_path(spec["reasonif_sft"])

print(f"Loading Base records: {base_path}")
with open(base_path, "r", encoding="utf-8") as f:
    base_records = [json.loads(line) for line in f if line.strip()]

print(f"Loading SFT records:  {sft_path}")
with open(sft_path, "r", encoding="utf-8") as f:
    sft_records = [json.loads(line) for line in f if line.strip()]

def get_record_key(r: Dict[str, Any]) -> str:
    if DATASET_NAME == "haskins_500":
        if "task" in r and "prompt_idx" in r:
            return f"{r['task']}:{r['prompt_idx']}"
        if "id" in r:
            parts = str(r["id"]).split(":")
            if len(parts) >= 2:
                try:
                    return f"{parts[0]}:{int(parts[1])}"
                except ValueError:
                    return str(r["id"])
            return str(r["id"])
        raise KeyError(f"Unable to extract Haskins key: {list(r.keys())}")
    if "dataset_index" in r:
        return str(r["dataset_index"])
    return str(r.get("id", r.get("key")))

base_map = {}
for r in base_records:
    k = get_record_key(r)
    if k in base_map:
        raise ValueError(f"Duplicate key detected in Base records: {k}")
    base_map[k] = r

sft_map = {}
for r in sft_records:
    k = get_record_key(r)
    if k in sft_map:
        raise ValueError(f"Duplicate key detected in SFT records: {k}")
    sft_map[k] = r

# Complete matching check
missing_in_sft = set(base_map.keys()) - set(sft_map.keys())
missing_in_base = set(sft_map.keys()) - set(base_map.keys())
if missing_in_sft or missing_in_base:
    raise ValueError(f"Incomplete matching! Missing in SFT: {len(missing_in_sft)}, Missing in Base: {len(missing_in_base)}")

matched_keys = sorted(list(base_map.keys()))
print(f"Successfully validated {len(matched_keys)} complete, unique paired records.")

paired_data = []
hasher = hashlib.sha256()

for k in matched_keys:
    b_item = base_map[k]
    s_item = sft_map[k]
    
    b_pass = extract_compliance_label(b_item)
    s_pass = extract_compliance_label(s_item)
    
    if s_pass and not b_pass:
        quadrant = "Group 1: SFT=PASS, Base=FAIL (Focal Signal)"
        q_code = "G1"
    elif s_pass and b_pass:
        quadrant = "Group 2: SFT=PASS, Base=PASS (Concordant Pass)"
        q_code = "G2"
    elif not s_pass and not b_pass:
        quadrant = "Group 3: SFT=FAIL, Base=FAIL (Concordant Fail)"
        q_code = "G3"
    else:
        quadrant = "Group 4: SFT=FAIL, Base=PASS (SFT Regression)"
        q_code = "G4"
        
    # Verify rendered chat prompt
    if "rendered_prompt" in s_item and s_item["rendered_prompt"]:
        prompt_text = s_item["rendered_prompt"]
    elif "full_prompt" in s_item and s_item["full_prompt"]:
        prompt_text = s_item["full_prompt"]
    else:
        prompt_text = s_item.get("prompt", "")
        
    sft_output = s_item.get("raw_output", s_item.get("text", ""))
    
    # Update input hash for cache validation
    hasher.update(k.encode("utf-8"))
    hasher.update(prompt_text.encode("utf-8"))
    hasher.update(sft_output.encode("utf-8"))
    
    entry = {
        "key": k,
        "task": b_item.get("task", b_item.get("source", "reasonif")),
        "prompt_idx": b_item.get("prompt_idx", b_item.get("dataset_index", -1)),
        "full_prompt": prompt_text,
        "sft_output": sft_output,
        "base_pass": b_pass,
        "sft_pass": s_pass,
        "quadrant": quadrant,
        "q_code": q_code,
        "reported_reasoning_tokens": s_item.get("reasoning_token_count", s_item.get("reasoning_tokens")),
        "reported_total_tokens": s_item.get("total_generated_tokens", s_item.get("output_tokens", s_item.get("generated_tokens"))),
    }
    
    # Preserve saved token IDs directly from SFT record if present
    if "prompt_token_ids" in s_item and "generated_token_ids" in s_item:
        entry["saved_prompt_token_ids"] = s_item["prompt_token_ids"]
        entry["saved_generated_token_ids"] = s_item["generated_token_ids"]
    else:
        entry["saved_prompt_token_ids"] = None
        entry["saved_generated_token_ids"] = None
        
    paired_data.append(entry)

computed_input_hash = hasher.hexdigest()
df_paired = pd.DataFrame(paired_data)
print(f"Input Verification Hash (SHA-256): {computed_input_hash}")
print("\n--- Verified Quadrant Breakdown ---")
summary = df_paired["quadrant"].value_counts().reset_index()
summary.columns = ["Quadrant", "Count"]
summary["Proportion (%)"] = (summary["Count"] / len(df_paired) * 100).round(2)
display(summary)


Loading Base records: /home/kyleliu789/workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_base/full_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl
Loading SFT records:  /home/kyleliu789/workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl
Successfully validated 300 complete, unique paired records.
Input Verification Hash (SHA-256): 73c4d35d02488ab2d32e6aa678d4172943a393666880e4f954ee3ddbbfeb5124

--- Verified Quadrant Breakdown ---


,Quadrant,Count,Proportion (%)
0,"Group 3: SFT=FAIL, Base=FAIL (Concordant Fail)",263,87.67
1,"Group 1: SFT=PASS, Base=FAIL (Focal Signal)",22,7.33
2,"Group 2: SFT=PASS, Base=PASS (Concordant Pass)",14,4.67
3,"Group 4: SFT=FAIL, Base=PASS (SFT Regression)",1,0.33


## 4. Reasoning Boundary Detection & Dual-Policy Forward Passes
1. **Token-Level Boundary Detection**:
   Inspects the prompt's reasoning state (whether `<think>` was already opened by the prompt) and searches for opening `<think>` and closing `</think>` tokens in generation. Reasoning terminates **strictly before** `</think>`.
2. **Float32 Precision**:
   Logits are converted via `.float()` prior to `F.log_softmax` and KL calculation.
3. **Comprehensive Cache Validation**:
   Checks model family, model revision, adapter revision, dataset name, measurement window, and the input SHA-256 hash before loading.

In [13]:
def detect_reasoning_token_bounds(tokenizer, p_ids: List[int], g_ids: List[int], prompt_text: str, raw_output: str) -> Tuple[int, int]:
    """
    Detects reasoning start and end indices strictly within g_ids using token IDs and prompt state.
    Returns (start_idx, end_idx) where g_ids[start_idx:end_idx] are reasoning tokens.
    Guarantees:
      1. If prompt already opened <think> without closing, start_idx = 0.
      2. If prompt did not open <think>, locates <think> in g_ids and sets start_idx after it.
      3. Reasoning stops strictly BEFORE </think>.
      4. If no <think> is present in prompt or generation, returns (0, 0) without misclassifying.
    """
    think_open_ids = tokenizer.encode("<think>", add_special_tokens=False)
    think_close_ids = tokenizer.encode("</think>", add_special_tokens=False)
    
    # 1. Determine if prompt already opened <think>
    prompt_has_open = ("<think>" in prompt_text)
    prompt_has_close = ("</think>" in prompt_text)
    
    if prompt_has_open:
        last_open_char = prompt_text.rfind("<think>")
        last_close_char = prompt_text.rfind("</think>") if prompt_has_close else -1
        prompt_reasoning_active = (last_open_char > last_close_char)
    else:
        prompt_reasoning_active = False
        for i in range(len(p_ids) - len(think_open_ids) + 1):
            if p_ids[i : i + len(think_open_ids)] == think_open_ids:
                prompt_reasoning_active = True
                
    # 2. Determine start_idx in generation
    if prompt_reasoning_active:
        start_idx = 0
    else:
        open_pos = -1
        for i in range(len(g_ids) - len(think_open_ids) + 1):
            if g_ids[i : i + len(think_open_ids)] == think_open_ids:
                open_pos = i
                break
        if open_pos != -1:
            start_idx = open_pos + len(think_open_ids)
        else:
            # Check token-by-token decoding
            curr_str = ""
            for i in range(len(g_ids)):
                curr_str += tokenizer.decode([g_ids[i]])
                if "<think>" in curr_str:
                    open_pos = i + 1
                    break
            if open_pos != -1:
                start_idx = open_pos
            else:
                # No think tags present; do not treat output as reasoning
                return 0, 0
                
    # 3. Determine end_idx in generation (must stop strictly before </think>)
    close_pos = -1
    for i in range(start_idx, len(g_ids) - len(think_close_ids) + 1):
        if g_ids[i : i + len(think_close_ids)] == think_close_ids:
            close_pos = i
            break
            
    if close_pos == -1:
        for i in range(start_idx, len(g_ids)):
            tok_text = tokenizer.decode([g_ids[i]])
            if "</think>" in tok_text:
                close_pos = i
                break
                
    if close_pos != -1:
        end_idx = close_pos
    else:
        end_idx = len(g_ids)
        
    return max(0, start_idx), max(start_idx, end_idx)

def compute_reasoning_kl(model, tokenizer, record: Dict[str, Any], max_reasoning_tokens: int = 512) -> Dict[str, Any]:
    """
    Computes per-token forward KL strictly along reasoning tokens in float32.
    """
    prompt_str = record["full_prompt"]
    raw_output = record["sft_output"]
    
    # Check if saved token IDs exist directly from SFT record
    if record.get("saved_prompt_token_ids") is not None and record.get("saved_generated_token_ids") is not None:
        p_ids = list(record["saved_prompt_token_ids"])
        g_ids = list(record["saved_generated_token_ids"])
        token_provenance = "sft_record_saved_token_ids"
    else:
        # Verified reconstruction
        if "<|im_start|>" not in prompt_str and "<|user|>" not in prompt_str:
            messages = [{"role": "user", "content": prompt_str}]
            prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
        p_ids = tokenizer.encode(prompt_str, add_special_tokens=True)
        g_ids = tokenizer.encode(raw_output, add_special_tokens=False)
        token_provenance = "verified_chat_reconstruction"
        
    # Detect exact reasoning boundaries
    r_start, r_end = detect_reasoning_token_bounds(tokenizer, p_ids, g_ids, prompt_str, raw_output)
    reasoning_ids = g_ids[r_start:r_end][:max_reasoning_tokens]
    
    if len(reasoning_ids) == 0:
        return {
            "reasoning_kl": [], "r_start": r_start, "r_end": r_end, 
            "total_r_tokens": 0, "provenance": token_provenance
        }
        
    eval_g_ids = g_ids[: r_start + len(reasoning_ids)]
    full_ids = p_ids + eval_g_ids
    input_ids = torch.tensor([full_ids], dtype=torch.long, device=device)
    prompt_len = len(p_ids)
    eval_gen_len = len(eval_g_ids)
    
    with torch.no_grad():
        out_sft = model(input_ids=input_ids)
        logits_sft = out_sft.logits[0, prompt_len - 1 : prompt_len - 1 + eval_gen_len, :]
        
        with model.disable_adapter():
            out_base = model(input_ids=input_ids)
            logits_base = out_base.logits[0, prompt_len - 1 : prompt_len - 1 + eval_gen_len, :]
            
        # Float32 numerical precision
        log_p = F.log_softmax(logits_sft.float(), dim=-1)
        log_q = F.log_softmax(logits_base.float(), dim=-1)
        p = torch.exp(log_p)
        
        full_kl = torch.sum(p * (log_p - log_q), dim=-1).cpu().numpy()
        reasoning_kl = full_kl[r_start : r_start + len(reasoning_ids)]
        
    return {
        "reasoning_kl": reasoning_kl.tolist(),
        "r_start": r_start,
        "r_end": r_end,
        "total_r_tokens": r_end - r_start,
        "measured_r_tokens": len(reasoning_ids),
        "provenance": token_provenance
    }

cache_file = Path(f"kl_cache_{MODEL_FAMILY}_{DATASET_NAME}.json")

if EXECUTE_FORWARD_PASSES:
    print(f"Loading Base model: {spec['base_model_id']} in bfloat16...")
    tokenizer = AutoTokenizer.from_pretrained(
        spec["base_model_id"], revision=spec["base_model_revision"], trust_remote_code=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        spec["base_model_id"],
        revision=spec["base_model_revision"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"Attaching LoRA Adapter: {spec['lora_adapter_id']}...")
    model = PeftModel.from_pretrained(
        base_model, spec["lora_adapter_id"], revision=spec["lora_adapter_revision"]
    )
    model.eval()
    
    cache_data = {
        "metadata": {
            "model_family": MODEL_FAMILY,
            "base_model_id": spec["base_model_id"],
            "base_model_revision": spec["base_model_revision"],
            "lora_adapter_id": spec["lora_adapter_id"],
            "lora_adapter_revision": spec["lora_adapter_revision"],
            "dataset_name": DATASET_NAME,
            "max_reasoning_tokens": MAX_REASONING_TOKENS,
            "measurement_window": f"first_{MAX_REASONING_TOKENS}_reasoning_tokens",
            "num_records": len(df_paired),
            "input_hash": computed_input_hash
        },
        "records": {}
    }
    
    print("Executing dual-policy forward passes over exact reasoning traces...")
    for idx, row in tqdm(df_paired.iterrows(), total=len(df_paired), desc="Computing Reasoning KL"):
        res = compute_reasoning_kl(model, tokenizer, row.to_dict(), max_reasoning_tokens=MAX_REASONING_TOKENS)
        cache_data["records"][row["key"]] = res
        
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=1)
    print(f"Saved verified measurement cache to {cache_file.resolve()}")
else:
    if not cache_file.exists():
        raise FileNotFoundError(
            f"Measurement cache '{cache_file}' not found.\n"
            f"To evaluate this dataset empirically, set EXECUTE_FORWARD_PASSES = True on a GPU VM."
        )
    print(f"Loading empirical measurement cache from {cache_file}...")
    with open(cache_file, "r", encoding="utf-8") as f:
        cache_data = json.load(f)
        
    # Rigorous cache validation
    meta = cache_data.get("metadata", {})
    required_validations = [
        ("model_family", MODEL_FAMILY),
        ("base_model_id", spec["base_model_id"]),
        ("base_model_revision", spec["base_model_revision"]),
        ("lora_adapter_id", spec["lora_adapter_id"]),
        ("lora_adapter_revision", spec["lora_adapter_revision"]),
        ("dataset_name", DATASET_NAME),
        ("max_reasoning_tokens", MAX_REASONING_TOKENS),
        ("input_hash", computed_input_hash)
    ]
    for key, expected_val in required_validations:
        actual_val = meta.get(key)
        if actual_val != expected_val:
            raise ValueError(
                f"Cache validation failed on '{key}': expected {expected_val!r}, but cache has {actual_val!r}. "
                f"The cache does not match the active model checkpoint, adapter revision, input prompts, or window. "
                f"Please set EXECUTE_FORWARD_PASSES = True to recompute forward passes."
            )

# Attach measured values strictly by verified key
reasoning_traces = []
r_lengths = []
for k in df_paired["key"]:
    if k not in cache_data["records"]:
        raise KeyError(f"Missing cached trace for key: {k}")
    rec = cache_data["records"][k]
    reasoning_traces.append(np.array(rec["reasoning_kl"], dtype=np.float32))
    r_lengths.append(rec.get("total_r_tokens", 0))

df_paired["reasoning_kl"] = reasoning_traces
df_paired["reasoning_token_count"] = r_lengths
print(f"Attached {len(df_paired)} verified reasoning traces. Median reasoning length: {np.median(r_lengths):.0f} tokens.")


FileNotFoundError: Measurement cache 'kl_cache_phi4_reasonif_300.json' not found.
To evaluate this dataset empirically, set EXECUTE_FORWARD_PASSES = True on a GPU VM.

## 5. Per-Task Breakdown & Question Dependence
Because Haskins repeats the same 50 questions across 10 tasks, pooling all 500 samples into a single independent t-test violates sample independence.
Here we evaluate **per-task distributions** and report descriptive statistics within each independent task condition.

In [14]:
early_r_kl = []
downstream_r_kl = []

for tr in df_paired["reasoning_kl"]:
    e_val = np.mean(tr[:10]) if len(tr) >= 10 else (np.mean(tr) if len(tr) > 0 else np.nan)
    d_val = np.mean(tr[10:100]) if len(tr) >= 100 else (np.mean(tr[10:]) if len(tr) > 10 else np.nan)
    early_r_kl.append(e_val)
    downstream_r_kl.append(d_val)

df_paired["early_reasoning_kl"] = early_r_kl
df_paired["downstream_reasoning_kl"] = downstream_r_kl

# Per-task aggregation
task_summary = df_paired.groupby(["task", "q_code"]).agg({
    "key": "count",
    "early_reasoning_kl": "mean",
    "downstream_reasoning_kl": "mean"
}).round(4).reset_index()

print("--- Per-Task Reasoning Divergence by Quadrant ---")
display(task_summary.pivot(index="task", columns="q_code", values="early_reasoning_kl"))


KeyError: 'reasoning_kl'

## 6. Candidate Prefix Window Analysis
We evaluate measured divergence across candidate prefix windows $k \in \{5, 10, 15, 20, 30, 50, 100\}$ reasoning tokens:

$$\text{Cumulative Reasoning KL}(k) = \sum_{t=0}^{k-1} D_{\text{KL}}^{\text{reasoning}}(t)$$

*Hypothesis-Generation Value*: Identifying token intervals where divergence stabilizes highlights candidate prefix cutoffs to evaluate in subsequent prefix intervention sweeps.

In [ ]:
prefix_k_list = [5, 10, 15, 20, 30, 50, 100]
window_rows = []

for k in prefix_k_list:
    for q in ["G1", "G2", "G3", "G4"]:
        sub = df_paired[df_paired["q_code"] == q]
        if len(sub) == 0:
            continue
            
        mean_k_vals = []
        frac_vals = []
        for tr in sub["reasoning_kl"]:
            if len(tr) >= k:
                mean_k_vals.append(np.mean(tr[:k]))
                tot = np.sum(tr)
                if tot > 0:
                    frac_vals.append(np.sum(tr[:k]) / tot)
                    
        if mean_k_vals:
            window_rows.append({
                "Prefix k": k,
                "Quadrant": q,
                "Mean KL (0..k)": np.mean(mean_k_vals),
                "% of Measured Window Divergence": np.mean(frac_vals) * 100
            })

df_k_summary = pd.DataFrame(window_rows)
display(df_k_summary.pivot(index="Prefix k", columns="Quadrant", values="Mean KL (0..k)").round(4))


## 7. Measured Publication Plots
All figures are generated strictly from **measured empirical data** without predetermined interpretations or synthetic curves.

In [ ]:
plot_dir = Path("kl_reasoning_plots")
plot_dir.mkdir(exist_ok=True)

horizon = 60
steps = np.arange(1, horizon + 1)

palette = {
    "G1": "#d62728", # Red
    "G2": "#2ca02c", # Green
    "G3": "#7f7f7f", # Gray
    "G4": "#1f77b4"  # Blue
}

quad_names = {
    "G1": "G1: SFT=1, Base=0 (Focal Signal)",
    "G2": "G2: SFT=1, Base=1 (Concordant Pass)",
    "G3": "G3: SFT=0, Base=0 (Concordant Fail)",
    "G4": "G4: SFT=0, Base=1 (SFT Regression)"
}

# ------------------------------------------------------------------------------
# FIGURE 1: Reasoning-Aligned Per-Token Trajectory
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 5.5))
has_valid_plot = False

for q in ["G1", "G2", "G3", "G4"]:
    sub = df_paired[df_paired["q_code"] == q]
    traces = [tr[:horizon] for tr in sub["reasoning_kl"] if len(tr) >= horizon]
    if len(traces) > 0:
        has_valid_plot = True
        mat = np.array(traces)
        mean_c = np.mean(mat, axis=0)
        se_c = stats.sem(mat, axis=0)
        
        plt.plot(steps, mean_c, label=f"{quad_names[q]} (n={len(traces)})", color=palette[q], lw=2.2)
        plt.fill_between(steps, mean_c - 1.96*se_c, mean_c + 1.96*se_c, color=palette[q], alpha=0.15)

if has_valid_plot:
    plt.axvline(x=10, color="black", linestyle="--", alpha=0.6, label="10-Token Reference Window")
    plt.title(f"Measured Reasoning-Aligned Forward KL Divergence ({MODEL_FAMILY.upper()} - {DATASET_NAME.upper()})")
    plt.xlabel("Reasoning Token Index ($t_{\mathrm{reasoning}}$)")
    plt.ylabel("Forward KL Divergence (nats)")
    plt.legend(frameon=True, facecolor="white")
    plt.tight_layout()
    plt.savefig(plot_dir / f"fig1_reasoning_kl_{MODEL_FAMILY}_{DATASET_NAME}.png", dpi=300)
    plt.show()

# ------------------------------------------------------------------------------
# FIGURE 2: Early vs Downstream Reasoning Boxplots
# ------------------------------------------------------------------------------
valid_df = df_paired.dropna(subset=["early_reasoning_kl", "downstream_reasoning_kl"]).copy()
if len(valid_df) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.boxplot(data=valid_df, x="q_code", y="early_reasoning_kl", palette=palette, ax=ax1, order=["G1", "G2", "G3", "G4"])
    ax1.set_title("Early Reasoning ($t_{\mathrm{reasoning}} \in [0, 9]$)")
    ax1.set_xlabel("Outcome Quadrant")
    ax1.set_ylabel("KL Divergence (nats)")
    
    sns.boxplot(data=valid_df, x="q_code", y="downstream_reasoning_kl", palette=palette, ax=ax2, order=["G1", "G2", "G3", "G4"])
    ax2.set_title("Downstream Reasoning ($t_{\mathrm{reasoning}} \in [10, 99]$)")
    ax2.set_xlabel("Outcome Quadrant")
    ax2.set_ylabel("KL Divergence (nats)")
    
    plt.tight_layout()
    plt.savefig(plot_dir / f"fig2_boxplots_{MODEL_FAMILY}_{DATASET_NAME}.png", dpi=300)
    plt.show()


## 8. Exact Token-Level Inspector (Actual Decoded Tokens)
Renders the actual decoded vocabulary tokens from the real evaluation record with measured float32 KL values.

In [ ]:
from IPython.display import HTML, display

def inspect_actual_record(tokenizer, record: Dict[str, Any], max_toks: int = 35):
    raw_output = record["sft_output"]
    prompt_str = record["full_prompt"]
    
    if record.get("saved_prompt_token_ids") is not None and record.get("saved_generated_token_ids") is not None:
        p_ids = list(record["saved_prompt_token_ids"])
        g_ids = list(record["saved_generated_token_ids"])
    else:
        p_ids = tokenizer.encode(prompt_str, add_special_tokens=True)
        g_ids = tokenizer.encode(raw_output, add_special_tokens=False)
        
    r_start, r_end = detect_reasoning_token_bounds(tokenizer, p_ids, g_ids, prompt_str, raw_output)
    tokens = [tokenizer.decode([tid]) for tid in g_ids[r_start : r_start + max_toks]]
    kl_vals = np.array(record["reasoning_kl"])[:len(tokens)]
    
    if len(kl_vals) == 0:
        print("No measured values available for this record.")
        return
        
    max_v = max(np.max(kl_vals), 0.5)
    
    html = f"<h4>Record Key: {record['key']} | Task: {record['task']} | {record['quadrant']}</h4>"
    html += "<p><i>Tokens displayed inside &lt;think&gt; reasoning window:</i></p>"
    html += "<div style='line-height: 2.4; font-family: monospace; font-size: 13px; padding: 12px; border: 1px solid #ddd; border-radius: 6px; background-color: #fafafa;'>"
    
    for tok, val in zip(tokens, kl_vals):
        norm = min(val / max_v, 1.0)
        r = int(255)
        g = int(240 * (1 - norm))
        b = int(200 * (1 - norm))
        tok_esc = tok.replace(" ", "&nbsp;").replace("\n", "<br/>&para;")
        html += f"<span style='background-color: rgb({r},{g},{b}); padding: 3px 5px; margin: 2px; border-radius: 3px;' title='KL: {val:.4f} nats'>{tok_esc}</span>"
        
    html += "</div>"
    display(HTML(html))

# Inspect first Group 1 record
g1_samples = df_paired[df_paired["q_code"] == "G1"]
if len(g1_samples) > 0 and "reasoning_kl" in df_paired.columns:
    sample = g1_samples.iloc[0].to_dict()
    if "tokenizer" in locals():
        inspect_actual_record(tokenizer, sample, max_toks=35)
    else:
        print(f"Record Key:      {sample['key']}")
        print(f"Task:            {sample['task']}")
        print(f"Early KL:        {sample.get('early_reasoning_kl', 'N/A')}")
        print(f"Downstream KL:   {sample.get('downstream_reasoning_kl', 'N/A')}")
        print(f"First 10 Values: {np.array(sample['reasoning_kl'])[:10]}")


## 9. Structured Export
Exports per-sample reasoning metrics and per-task aggregations to CSV.

In [ ]:
out_dir = Path("kl_reasoning_results")
out_dir.mkdir(exist_ok=True)

export_cols = [
    "key", "task", "prompt_idx", "q_code", "quadrant", 
    "base_pass", "sft_pass", "early_reasoning_kl", "downstream_reasoning_kl", "reasoning_token_count"
]
df_export = df_paired[export_cols].copy()
df_export.to_csv(out_dir / f"per_sample_reasoning_kl_{MODEL_FAMILY}_{DATASET_NAME}.csv", index=False)

task_export = df_paired.groupby(["task", "q_code"]).agg({
    "key": "count",
    "early_reasoning_kl": ["mean", "std"],
    "downstream_reasoning_kl": ["mean", "std"],
    "reasoning_token_count": ["median"]
}).round(4)
task_export.to_csv(out_dir / f"per_task_reasoning_kl_{MODEL_FAMILY}_{DATASET_NAME}.csv")

print(f"Exported empirical data to {out_dir.resolve()}:")
print(f"  - per_sample_reasoning_kl_{MODEL_FAMILY}_{DATASET_NAME}.csv")
print(f"  - per_task_reasoning_kl_{MODEL_FAMILY}_{DATASET_NAME}.csv")
display(task_export.head(12))
